In [3]:
import serial
import time
import binascii
from datetime import datetime
import os

class ModbusSniffer:
    def __init__(self, port, baudrate=9600, log_file=None):
        """
        Инициализация сниффера Modbus RTU
        
        Args:
            port: COM-порт (например, 'COM3' или '/dev/ttyUSB0')
            baudrate: скорость передачи
            log_file: путь к файлу лога (если None, создается автоматически)
        """
        self.port = port
        self.baudrate = baudrate
        self.ser = None
        self.running = False
        self.packet_count = 0
        
        # Создаем имя файла лога, если не указано
        if log_file is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            self.log_file = f"modbus_sniff_{timestamp}.log"
        else:
            self.log_file = log_file
        
        # Создаем директорию для логов, если её нет
        log_dir = os.path.dirname(self.log_file)
        if log_dir and not os.path.exists(log_dir):
            os.makedirs(log_dir)
    
    def connect(self):
        """Подключение к порту"""
        try:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=self.baudrate,
                bytesize=8,
                parity='N',
                stopbits=1,
                timeout=0.5
            )
            print(f"✅ Подключен к {self.port} (скорость: {self.baudrate})")
            print(f"📝 Лог-файл: {self.log_file}")
            return True
        except Exception as e:
            print(f"❌ Ошибка подключения: {e}")
            return False
    
    def close(self):
        """Закрытие порта"""
        if self.ser:
            self.ser.close()
            print("✅ Порт закрыт")
    
    def _write_log(self, message):
        """Запись в лог-файл"""
        try:
            with open(self.log_file, 'a', encoding='utf-8') as f:
                f.write(message + '\n')
        except Exception as e:
            print(f"⚠️ Ошибка записи в лог: {e}")
    
    def _get_timestamp(self):
        """Получение временной метки"""
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
    
    def _parse_modbus_frame(self, data):
        """
        Простой парсинг Modbus кадра
        """
        if len(data) < 5:
            return "Неполный кадр"
        
        info = []
        info.append(f"Длина: {len(data)} байт")
        
        # Адрес и функция
        slave_addr = data[0]
        func_code = data[1]
        info.append(f"Адрес: 0x{slave_addr:02X}")
        info.append(f"Функция: 0x{func_code:02X}")
        
        # Проверка на ошибку
        if func_code & 0x80:
            info.append(f"⚠️ ОШИБКА! Код: 0x{data[2]:02X}")
            return info
        
        # Если это ответ с данными
        if len(data) >= 3:
            data_length = data[2]
            info.append(f"Длина данных: {data_length} байт")
            
            if len(data) >= 3 + data_length:
                data_bytes = data[3:3+data_length]
                info.append(f"Данные: {binascii.hexlify(data_bytes).upper().decode()}")
                
                # Попытка интерпретации
                if data_length >= 2:
                    value = (data_bytes[0] << 8) | data_bytes[1]
                    info.append(f"  UINT16: {value} (0x{value:04X})")
                
                if data_length >= 4:
                    # Float (пробуем разные варианты)
                    import struct
                    try:
                        f_be = struct.unpack('>f', data_bytes[:4])[0]
                        info.append(f"  Float (BE): {f_be:.6f}")
                    except: pass
                    
                    try:
                        f_le = struct.unpack('<f', data_bytes[:4])[0]
                        info.append(f"  Float (LE): {f_le:.6f}")
                    except: pass
        
        return info
    
    def start_sniffing(self):
        """Запуск сниффинга"""
        if not self.ser:
            print("❌ Нет подключения к порту")
            return
        
        self.running = True
        print("\n" + "="*70)
        print("🔍 НАЧАЛО СНИФФИНГА")
        print("="*70)
        print("Нажмите Ctrl+C для остановки")
        print("="*70 + "\n")
        
        # Записываем начало сессии в лог
        header = [
            "="*70,
            f"СНИФФИНГ MODBUS RTU",
            f"Порт: {self.port}",
            f"Скорость: {self.baudrate}",
            f"Начало: {self._get_timestamp()}",
            "="*70
        ]
        self._write_log("\n".join(header))
        
        try:
            while self.running:
                if self.ser.in_waiting > 0:
                    # Читаем данные
                    raw_data = self.ser.read(self.ser.in_waiting)
                    self.packet_count += 1
                    
                    # Формируем информацию
                    timestamp = self._get_timestamp()
                    hex_data = binascii.hexlify(raw_data).upper().decode()
                    
                    # Вывод в консоль
                    print(f"\n📦 Пакет #{self.packet_count} [{timestamp}]")
                    print(f"HEX: {hex_data}")
                    
                    # Парсинг
                    info = self._parse_modbus_frame(raw_data)
                    if isinstance(info, list):
                        for line in info:
                            print(f"  {line}")
                    else:
                        print(f"  {info}")
                    
                    # Запись в лог
                    log_entry = [
                        f"\nПакет #{self.packet_count} [{timestamp}]",
                        f"HEX: {hex_data}",
                        f"Длина: {len(raw_data)} байт"
                    ]
                    
                    if isinstance(info, list):
                        log_entry.extend([f"  {line}" for line in info])
                    else:
                        log_entry.append(f"  {info}")
                    
                    self._write_log("\n".join(log_entry))
                
                time.sleep(0.01)
                
        except KeyboardInterrupt:
            print("\n\n⏹️ Остановка сниффинга...")
        except Exception as e:
            print(f"❌ Ошибка: {e}")
        finally:
            self.running = False
            self.close()
            
            # Записываем завершение сессии
            footer = [
                "="*70,
                f"ЗАВЕРШЕНИЕ СНИФФИНГА",
                f"Всего пакетов: {self.packet_count}",
                f"Время: {self._get_timestamp()}",
                f"Лог-файл: {self.log_file}",
                "="*70
            ]
            self._write_log("\n".join(footer))
            print(f"\n📝 Лог сохранен в: {self.log_file}")
            print(f"📊 Всего пакетов: {self.packet_count}")

# Использование
if __name__ == "__main__":
    # Создаем сниффер
    sniffer = ModbusSniffer(
        port='COM5',
        baudrate=57600, #9600,
        log_file='modbus_log.txt'  # Или None для автоматического имени
    )
    
    if sniffer.connect():
        sniffer.start_sniffing()

✅ Подключен к COM5 (скорость: 9600)
📝 Лог-файл: modbus_log.txt

🔍 НАЧАЛО СНИФФИНГА
Нажмите Ctrl+C для остановки



⏹️ Остановка сниффинга...
✅ Порт закрыт

📝 Лог сохранен в: modbus_log.txt
📊 Всего пакетов: 0


In [1]:
import serial
import time
import binascii
from datetime import datetime
import os
import threading
import queue

class ProtocolSniffer:
    def __init__(self, port, baudrate=9600, log_dir='logs', 
                 start_marker=b'\x55\x55', max_log_size=10*1024*1024):
        """
        Универсальный сниффер с поиском маркера начала пакета
        
        Args:
            port: COM-порт
            baudrate: скорость
            log_dir: директория для логов
            start_marker: маркер начала пакета (по умолчанию 0x55 0xAA)
            max_log_size: максимальный размер лог-файла
        """
        self.port = port
        self.baudrate = baudrate
        self.log_dir = log_dir
        self.start_marker = start_marker
        self.max_log_size = max_log_size
        self.ser = None
        self.running = False
        self.packet_count = 0
        self.buffer = bytearray()  # Буфер для накопления данных
        self.log_queue = queue.Queue()
        self.log_thread = None
        
        # Создаем директорию
        if not os.path.exists(log_dir):
            os.makedirs(log_dir)
        
        self.current_log_file = self._get_log_filename()
    
    def _get_log_filename(self):
        """Получение имени файла лога"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        return os.path.join(self.log_dir, f"sniff_{timestamp}.log")
    
    def _write_log(self, message):
        """Асинхронная запись в лог"""
        self.log_queue.put(message)
    
    def _log_worker(self):
        """Поток для записи логов"""
        while self.running or not self.log_queue.empty():
            try:
                message = self.log_queue.get(timeout=1)
                with open(self.current_log_file, 'a', encoding='utf-8') as f:
                    f.write(message + '\n')
                    f.flush()
                self.log_queue.task_done()
            except queue.Empty:
                continue
            except Exception as e:
                print(f"⚠️ Ошибка записи лога: {e}")
    
    def connect(self):
        """Подключение к порту"""
        try:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=self.baudrate,
                bytesize=8,
                parity='N',
                stopbits=1,
                timeout=0.5
            )
            print(f"✅ Подключен к {self.port} (скорость: {self.baudrate})")
            print(f"📁 Директория логов: {self.log_dir}")
            print(f"🔍 Маркер начала пакета: {binascii.hexlify(self.start_marker).upper().decode()}")
            return True
        except Exception as e:
            print(f"❌ Ошибка подключения: {e}")
            return False
    
    def close(self):
        """Закрытие порта"""
        if self.ser:
            self.ser.close()
    
    def _find_packets(self, data):
        """
        Поиск пакетов в потоке данных по стартовому маркеру
        Возвращает список найденных пакетов и остаток данных
        """
        packets = []
        remaining = bytearray()
        
        # Добавляем новые данные в буфер
        self.buffer.extend(data)
        
        # Ищем маркер в буфере
        marker_len = len(self.start_marker)
        pos = 0
        
        while pos < len(self.buffer):
            # Ищем маркер
            marker_pos = self.buffer.find(self.start_marker, pos)
            
            if marker_pos == -1:
                # Маркер не найден, сохраняем остаток
                remaining = self.buffer[pos:]
                break
            
            # Нашли маркер
            packet_start = marker_pos
            
            # Ищем следующий маркер (конец пакета)
            next_marker_pos = self.buffer.find(self.start_marker, packet_start + marker_len)
            
            if next_marker_pos == -1:
                # Следующий маркер не найден, ждем еще данных
                remaining = self.buffer[packet_start:]
                break
            
            # Извлекаем пакет (от маркера до следующего маркера)
            packet = self.buffer[packet_start:next_marker_pos]
            packets.append(packet)
            
            # Продолжаем поиск
            pos = next_marker_pos
        
        # Обновляем буфер
        self.buffer = remaining
        
        return packets
    
    def _parse_packet(self, packet):
        """
        Анализ пакета (универсальный)
        """
        info = []
        info.append(f"Длина: {len(packet)} байт")
        if len(packet) == 10:
            # Первые 2 байта - маркер
            #info.append(f"Маркер: {binascii.hexlify(packet[0:2]).upper().decode()}")        
            info.append(f"куда: 0x{packet[2]:02X}, от кого: 0x{packet[3]:02X}, счет: 0x{packet[4]:02X}, Дл дан: 0x{packet[5]:02X}")                   
            #info.append(f"от кого: 0x{packet[3]:02X}")        
            #info.append(f"счет: 0x{packet[4]:02X}")    
            #info.append(f"Дл дан: 0x{packet[5]:02X}")
            info.append(f"Функ: 0x{packet[7]:02X}, Адр: 0x{packet[9]:02X}{packet[8]:02X}, Разм: 0x{packet[11]:02X}{packet[10]:02X},CRC: 0x{packet[12]:02X}")
            #info.append(f"Адр: 0x{packet[9]:02X} 0x{packet[8]:02X}")
            #info.append(f"Размер: 0x{packet[11]:02X} 0x{packet[10]:02X}")
            #info.append(f"CRC: 0x{packet[12]:02X}")
            # Проверяем, не является ли третий байт длиной
            #if len(packet) >= 4 and packet[2] + 3 == len(packet):
                #info.append(f"  Похоже на длину: {packet[2]} байт")
        # Показываем первые несколько байт
        max_show = min(16, len(packet))
        info.append(f"Первые {max_show} байт: {binascii.hexlify(packet[:max_show]).upper().decode()}")
        if len(packet) != 10:
            info.append(f"куда: 0x{packet[2]:02X}, от кого: 0x{packet[3]:02X}, счет: 0x{packet[4]:02X}, Дл дан: 0x{packet[5]:02X}")
            info.append(f"data: 0x{packet[7]:02X} {packet[8]:02X} {packet[9]:02X} {packet[10]:02X} {packet[11]:02X}")
            # Показываем первые несколько байт
        max_show = min(16, len(packet))
        info.append(f"Первые {max_show} байт: {binascii.hexlify(packet[:max_show]).upper().decode()}")
        if len(packet) > 16:
            info.append(f"... и еще {len(packet) - 16} байт")
        
        return info
    
    def start_sniffing(self):
        """Запуск сниффинга"""
        if not self.ser:
            print("❌ Нет подключения")
            return
        
        self.running = True
        self.packet_count = 0
        self.buffer = bytearray()
        
        # Запускаем поток записи логов
        self.log_thread = threading.Thread(target=self._log_worker, daemon=True)
        self.log_thread.start()
        
        print("\n" + "="*70)
        print("🔍 НАЧАЛО СНИФФИНГА (ПОИСК ПАКЕТОВ ПО МАРКЕРУ)")
        print("="*70)
        print(f"Порт: {self.port}")
        print(f"Скорость: {self.baudrate}")
        print(f"Маркер: {binascii.hexlify(self.start_marker).upper().decode()}")
        print("Нажмите Ctrl+C для остановки")
        print("="*70 + "\n")
        
        # Записываем начало
        header = [
            "="*70,
            "СНИФФИНГ ПРОТОКОЛА (поиск маркера)",
            f"Порт: {self.port}",
            f"Скорость: {self.baudrate}",
            f"Маркер: {binascii.hexlify(self.start_marker).upper().decode()}",
            f"Начало: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            "="*70
        ]
        self._write_log("\n".join(header))
        
        try:
            while self.running:
                if self.ser.in_waiting > 0:
                    # Читаем данные
                    raw_data = self.ser.read(self.ser.in_waiting)
                    
                    # Ищем пакеты
                    packets = self._find_packets(raw_data)
                    
                    # Обрабатываем найденные пакеты
                    for packet in packets:
                        self.packet_count += 1
                        timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
                        hex_data = binascii.hexlify(packet).upper().decode()
                        
                        # Вывод в консоль
                        print(f"\n📦 Пакет #{self.packet_count} [{timestamp}]")
                        print(f"HEX: {hex_data}")
                        print(f"Длина: {len(packet)} байт")
                        
                        # Анализ
                        info = self._parse_packet(packet)
                        for line in info:
                            print(f"  {line}")
                        
                        # Запись в лог
                        log_entry = [
                            f"\nПакет #{self.packet_count} [{timestamp}]",
                            f"HEX: {hex_data}",
                            f"Длина: {len(packet)} байт",
                            *[f"  {line}" for line in info]
                        ]
                        self._write_log("\n".join(log_entry))
                
                time.sleep(0.01)
                
        except KeyboardInterrupt:
            print("\n\n⏹️ Остановка сниффинга...")
        except Exception as e:
            print(f"❌ Ошибка: {e}")
        finally:
            self.running = False
            self.close()
            
            # Ждем завершения записи логов
            self.log_queue.join()
            
            # Записываем завершение
            footer = [
                "="*70,
                "ЗАВЕРШЕНИЕ СНИФФИНГА",
                f"Всего пакетов: {self.packet_count}",
                f"Время: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
                f"Лог-файл: {self.current_log_file}",
                "="*70
            ]
            self._write_log("\n".join(footer))
            self.log_queue.join()
            
            print(f"\n📝 Лог сохранен в: {self.current_log_file}")
            print(f"📊 Всего перехвачено пакетов: {self.packet_count}")

# Использование
if __name__ == "__main__":
    sniffer = ProtocolSniffer(
        port='COM5',
        baudrate=57600, #9600,
        log_dir='logs',
        start_marker=b'\x55\x55'  # Два байта 0x55 0xAA
    )
    
    if sniffer.connect():
        sniffer.start_sniffing()

✅ Подключен к COM5 (скорость: 57600)
📁 Директория логов: logs
🔍 Маркер начала пакета: 5555

🔍 НАЧАЛО СНИФФИНГА (ПОИСК ПАКЕТОВ ПО МАРКЕРУ)
Порт: COM5
Скорость: 57600
Маркер: 5555
Нажмите Ctrl+C для остановки


📦 Пакет #1 [11:55:03.910]
HEX: 5555FF011D050008C4000E00FC
Длина: 13 байт
  Длина: 13 байт
  Первые 13 байт: 5555FF011D050008C4000E00FC
  куда: 0xFF, от кого: 0x01, счет: 0x1D, Дл дан: 0x05
  data: 0x08 C4 00 0E 00
  Первые 13 байт: 5555FF011D050008C4000E00FC

📦 Пакет #2 [11:55:03.941]
HEX: 555501FF1D0E00000008000000070600000000000040
Длина: 22 байт
  Длина: 22 байт
  Первые 16 байт: 555501FF1D0E00000008000000070600
  куда: 0x01, от кого: 0xFF, счет: 0x1D, Дл дан: 0x0E
  data: 0x00 00 08 00 00
  Первые 16 байт: 555501FF1D0E00000008000000070600
  ... и еще 6 байт

📦 Пакет #3 [11:55:03.952]
HEX: 5555FF011E050008D200550052
Длина: 13 байт
  Длина: 13 байт
  Первые 13 байт: 5555FF011E050008D200550052
  куда: 0xFF, от кого: 0x01, счет: 0x1E, Дл дан: 0x05
  data: 0x08 D2 00 55 00
  Первые